In [ ]:
import math 
def parse_verilog_number(value, width=64):
    """
    Accepts:
      - 64'sd123
      - -64'sd123
      - 64'h1a2b
      - -64'h1a2b
      - 0x1a2b
      - plain decimal like 123
    Returns a signed Python int.
    """
    s = str(value).strip().replace("_", "")
    negative = False

    if s.startswith("-"):
        negative = True
        s = s[1:]

    low = s.lower()

    if "'sd" in low:
        number = int(low.split("'sd", 1)[1], 10)
        return -number if negative else number

    if "'sh" in low or "'h" in low:
        number = int(low.split("'h", 1)[1], 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    if low.startswith("0x"):
        number = int(low, 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    number = int(low, 10)
    return -number if negative else number
    
def parse_verilog_number(value, width=64):
    """
    Accepts:
      - 64'sd123
      - -64'sd123
      - 64'h1a2b
      - -64'h1a2b
      - 0x1a2b
      - plain decimal like 123
    Returns a signed Python int.
    """
    s = str(value).strip().replace("_", "")
    negative = False

    if s.startswith("-"):
        negative = True
        s = s[1:]

    low = s.lower()

    if "'sd" in low:
        number = int(low.split("'sd", 1)[1], 10)
        return -number if negative else number

    if "'sh" in low or "'h" in low:
        number = int(low.split("'h", 1)[1], 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    if low.startswith("0x"):
        number = int(low, 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    number = int(low, 10)
    return -number if negative else number

def signed_int_to_verilog_hex(value, width=64):
    mask = (1 << width) - 1
    unsigned_value = value & mask
    hex_digits = width // 4
    return f"{width}'h{unsigned_value:0{hex_digits}x}"


def parse_verilog_hex(hex_str):
    s = hex_str.lower().replace("0x", "").replace("64'h", "").replace("_", "")
    val = int(s, 16)
    if val >= 2**63:
        val -= 2**64
    return val

def signed64_to_hex(value):
    return signed_int_to_verilog_hex(value, width=64)


def fixed_to_float(value, frac_bits):
    return value / float(1 << frac_bits)


def q_fixed_to_float(value, frac_bits):
    return fixed_to_float(value, frac_bits)


def float_to_q_fixed(x, frac_bits, width=64):
    scaled = int(round(x * (1 << frac_bits)))
    mask = (1 << width) - 1
    scaled &= mask
    if scaled >= (1 << (width - 1)):
        scaled -= (1 << width)
    return scaled

def q3_61_hex_to_float(hex_str, frac_bits=61, width=64):
    val = parse_verilog_number(hex_str, width)
    return fixed_to_float(val, frac_bits)

def float_to_q_fixed_local(x, frac_bits, width=64):
    scaled = int(round(x * (1 << frac_bits)))
    mask = (1 << width) - 1
    scaled &= mask
    if scaled >= (1 << (width - 1)):
        scaled -= (1 << width)
    return scaled

def sincos_to_qhex(value_str, frac_bits=61, width=64):
    signed_int = parse_verilog_number(value_str, width)
    val_float = fixed_to_float(signed_int, frac_bits)
    s = math.sin(val_float)
    c = math.cos(val_float)
    s_q = float_to_q_fixed_local(s, frac_bits, width)
    c_q = float_to_q_fixed_local(c, frac_bits, width)
    return {
        "input_float": val_float,
        "sin_float": s,
        "sin_hex": signed_int_to_verilog_hex(s_q, width),
        "cos_float": c,
        "cos_hex": signed_int_to_verilog_hex(c_q, width),
    }

In [ ]:
def mul_q3_61_hex(a_hex, b_hex, width=64, frac_bits=61, rounding=False):
    # Reuse your existing helper if available
    a = parse_verilog_number(a_hex, width)
    b = parse_verilog_number(b_hex, width)

    # Multiply in full precision (Q6.122)
    prod = a * b

    # Optional rounding before shifting
    if rounding:
        prod += (1 << (frac_bits - 1)) if prod >= 0 else -(1 << (frac_bits - 1))

    # Convert back to Q3.61 by shifting right 61 bits
    q = prod >> frac_bits  # arithmetic shift for signed ints in Python

    # Wrap to signed 64-bit range
    mask = (1 << width) - 1
    q &= mask
    if q >= (1 << (width - 1)):
        q -= (1 << width)

    return signed_int_to_verilog_hex(q, width)
# comput the sin and cos and represent result in hex format into Q3.61 and Q4.60 
FRAC_BITS = 61  # Q3.61
WIDTH = 64

a = "64'h19518bebead3c500"  # +1.0 in Q3.61
b = "64'he2973c6b0f92a900"  # +1.0 in Q3.61
result = mul_q3_61_hex(a, b)
print(result)  # expect +1.0 -> 64'h2000000000000000
print(q3_61_hex_to_float(result))  # 1.0


64'he8bb356cb06e071b
